In [1]:
# import packages
import pandas as pd
import os, glob

In [2]:
# output directory
out_dir = "/private10/Projects/Efi/CRG/GBM/SplicingAnalysis/SplicingEvents/_forNEanalysis/"
# input file(s) - '*_merged_rank{}_aff{}.csv'
file_paths = glob.glob(os.path.join(out_dir,'*',f'*_merged_rank*'))
# define control and treatments groups
control_group_names = list(set([os.path.basename(file).split('_')[0] for file in file_paths]))
treatment_group_names = list(set([os.path.basename(file).split('_')[1] for file in file_paths]))
# print for assurance
print("Out Dir: ", out_dir, '\n', 'Files: ', file_paths, '\n', 'Controls: ', control_group_names, '\n','Treatments: ', treatment_group_names)


Out Dir:  /private10/Projects/Efi/CRG/GBM/SplicingAnalysis/SplicingEvents/_forNEanalysis/ 
 Files:  ['/private10/Projects/Efi/CRG/GBM/SplicingAnalysis/SplicingEvents/_forNEanalysis/DMSO_vs_H3B8800/DMSO_H3B8800_merged_rank0.5_aff50.0.csv'] 
 Controls:  ['DMSO'] 
 Treatments:  ['H3B8800']


In [3]:
# merge all files into one df and remove duplicates
dataframes = [pd.read_csv(file) for file in file_paths]
combined_dataframe = pd.concat(dataframes, ignore_index=True)
combined_dataframe_noDups = combined_dataframe.drop_duplicates()

In [4]:
# melt the data according to rank and nM
id_cols = ['Group','Splicing Event','Peptide','ID','SplicingIndex','Exon.Type' ,'Reference.Transcript','Avg.PSI_PeptideSource','Avg.PSI_OtherGroup','Avg.TPM_PeptideSource','Avg.TPM_OtherGroup'] 
data = combined_dataframe_noDups
df_rank = data.melt(id_vars=id_cols, 
                value_vars= data.columns[data.columns.str.endswith('_Rank')],
                var_name='HLA_Rank', 
                value_name='Rank').dropna(subset=['Rank'])
df_nM = data.melt(id_vars=id_cols, 
                value_vars= data.columns[data.columns.str.endswith('_nM')],
                var_name='HLA_nM', 
                value_name='nM').dropna(subset=['nM'])
df_rank['HLA'] = df_rank['HLA_Rank'].str.split('_').str[0]
df_nM['HLA'] = df_nM['HLA_nM'].str.split('_').str[0] 

df_rank.drop(columns=['HLA_Rank'], inplace=True)
df_nM.drop(columns=['HLA_nM'], inplace=True)

# Rearrange columns in each melted DataFrame
df_rank = df_rank[id_cols+['HLA', 'Rank']]
df_nM = df_nM[id_cols+['HLA', 'nM']]

df_merged = pd.merge(df_rank, df_nM, on=id_cols+['HLA'], how='inner')

In [5]:
# Identify peptides in both control and treatments
control_peptides = set(df_merged[df_merged['Group'].isin(control_group_names)]['Peptide'])
treatment_peptides = set(df_merged[df_merged['Group'].isin(treatment_group_names)]['Peptide'])
common_peptides = control_peptides.intersection(treatment_peptides)
# Filter the data frame
filtered_df = df_merged[~(df_merged['Peptide'].isin(common_peptides))]

In [6]:
# save the file
out_file = os.path.join(out_dir, 'NovelStrongBindingEpitopes_noDups.csv')
filtered_df.to_csv(out_file, index=False)